In [1]:
!nvidia-smi

Mon Feb 17 13:50:48 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.54.03              Driver Version: 535.54.03    CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100 80GB PCIe          Off | 00000000:17:00.0 Off |                    0 |
| N/A   59C    P0              74W / 300W |  17970MiB / 81920MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [2]:
import torch
print(torch.__version__)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/usr/local/lib/python3.10/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.10/dist-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/usr/local/lib/python3.10/dist-packages/ipykernel/kernelapp.p

2.1.0+cu121


In [ ]:
import os
import torch
!huggingface-cli login --token
# Limit PyTorch to only use GPUs 0 and 1
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [ ]:
import os
import random
import functools
import csv
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import evaluate

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, classification_report, balanced_accuracy_score, accuracy_score

from scipy.stats import pearsonr
from datasets import Dataset, DatasetDict
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
# !bash ../../../../connect.sh


In [ ]:
import pandas as pd
data = pd.read_csv("./Data/shakespearrean.csv")
data.head()

In [ ]:
data.info()

In [ ]:
data.columns

## Prompt Generation

In [ ]:
def prompt_generation(row,col):
    question = row["Malicious Questions"]
    response = row[col]
    message = [
                {"role": "user", "content": question},
                {"role": "assistant", "content": response},
                ]
    return message
    

## Model Loading

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "./models/llama-guard-3-8b"
device = "cuda"
dtype = torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map=device)



In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit = True, # enable 4-bit quantization
    bnb_4bit_quant_type = 'nf4', # information theoretically optimal dtype for normally distributed weights
    bnb_4bit_use_double_quant = True, # quantize quantized weights //insert xzibit meme
    bnb_4bit_compute_dtype = torch.bfloat16 # optimized fp format for ML
)

In [ ]:
lora_config = LoraConfig(
    r = 32,  # Reduced rank dimension
    lora_alpha = 32,  # Increased scaling factor
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], # Limited target layers
    lora_dropout = 0.05,
    bias = 'none',
    task_type = 'SEQ_CLS'
)




In [ ]:
label_map = {"Unsafe":0, "Safe":1}


In [ ]:
model_hf = AutoModelForSequenceClassification.from_pretrained(
    "Sravanigunnu/sequence_classification",
    quantization_config=quantization_config,
    num_labels=len(label_map),
    device_map="auto"
) 

In [ ]:
model_hf.config.pad_token_id = tokenizer.pad_token_id
model_hf.config.use_cache = False
model_hf.config.pretraining_tp = 1
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.pad_token = tokenizer.eos_token 

In [ ]:
from tqdm import tqdm
import numpy as np
def make_predictions(model, df, src_col,col_name):


  # Convert summaries to a list
  sentences = df[src_col].tolist()

  # Define the batch size
  batch_size = 32  # You can adjust this based on your system's memory capacity
  model.config.pad_token_id = tokenizer.pad_token_id
  # Initialize an empty list to store the model outputs
  all_outputs = []
  
  # Process the sentences in batches
  for i in tqdm(range(0, len(sentences), batch_size)):
      # Get the batch of sentences
      batch_sentences = sentences[i:i + batch_size]

      # Tokenize the batch
      inputs = tokenizer(batch_sentences, return_tensors="pt", padding=True, truncation=True, max_length=512)

      # Move tensors to the device where the model is (e.g., GPU or CPU)
      inputs = {k: v.to('cuda' if torch.cuda.is_available() else 'cpu') for k, v in inputs.items()}

      # Perform inference and store the logits
      with torch.no_grad():
          outputs = model(**inputs)
          all_outputs.append(outputs['logits'])

  final_outputs = torch.cat(all_outputs, dim=0)
  return final_outputs
  df[col_name]=final_outputs.argmax(axis=1).cpu().numpy()
  op = final_outputs.argmax(axis=1).cpu().numpy()
  # df['predictions']=df['predictions'].apply(lambda l:category_map[l])
  return df

In [ ]:
def moderate(chat,model):
    input_ids = tokenizer.apply_chat_template(chat, return_tensors="pt").to(device)
    output = model.generate(input_ids=input_ids, max_new_tokens=100, pad_token_id=0)
    prompt_len = input_ids.shape[-1]
    return tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True)

In [ ]:
def get_predicted_label(row,col,model):
    """Generate prediction using Llama-Guard and clean the output"""
    chat = prompt_generation(row,col)
    raw_output = moderate(chat,model).strip().lower().replace("\n", "")  # Normalize output
    # print(row["Num"],end= " ")
    return "Unsafe" if "unsafe" in raw_output else "Safe"

In [ ]:
# target_col_names = ['response_eng', 'response_te', 'response_te_eng',
#        'response_hi', 'response_hi_eng', 'response_mr', 'response_mr_eng',
#        'response_be', 'response_be_eng']
target_col_names = ["shakespearean_response_to_english"]

In [ ]:
predict_columns = ["Pred_label_res_"+col_name for col_name in target_col_names]
predict_columns

In [ ]:
df = data.copy()

In [ ]:
final_outputs = make_predictions(model_hf, df, target_col_names[0], col_name=predict_columns[0])
final_outputs 

In [ ]:
col_name=predict_columns[0]
df[col_name]=final_outputs.argmax(axis=1).cpu().numpy()

In [ ]:
df.head()

In [ ]:
target_col_name =  target_col_names[0]
pred_col = "base"+predict_columns[0]

# Replace with the actual column name if different
df[pred_col] = df.apply(lambda row: get_predicted_label(row, target_col_name,model), axis=1)

In [ ]:
df.info()

In [ ]:
df.to_csv("shakespearrean_results.csv", index= False)

In [ ]:
col_name = predict_columns[0]
non_asr = df[col_name].sum()
asr_final = (len(df) - non_asr )/len(df)
asr_final

In [ ]:
df.info()

In [ ]:
unsafe_count = (df["basePred_label_res_shakespearean_response_to_english"] == "Unsafe").sum()
print(unsafe_count)
# Count total data points
total_count = len(df)

# Calculate attack success rate
attack_success_rate = unsafe_count / total_count if total_count > 0 else 0

# Print the result
print(f"Attack Success Rate: {attack_success_rate:.2}") 

In [ ]:
df.to_csv("gemma_2_9b_it_role_play.csv",index=False)